# Kickbase percentile rating × odds × lineup score

Run derived-analysis notebook 10 first. This notebook selects its newest export by `generated_at` and uses its one-decimal per-match ratings in the shared overall odds/lineup workflow.

Team-match weights are 28%, 24%, 20%, 16%, 12%, newest first. Missing or excluded appearances are omitted and remaining weights renormalized without shifting slots. The weighted rating is rounded half-up to one decimal before scoring. The derived notebook now exports this same weighted average; this notebook recomputes it from individual match ratings so older exports remain compatible and weights are never applied twice. Persisted fallback flags are displayed when loading ratings.

Score = multiplier × Kickbase percentile rating × expected match points × starting chance. Odds, identity matching, matchday/source confirmation, injury handling, and lineup alternatives follow the existing overall rating notebook. The export uses `kickbase_percentile_rating`, a distinct method filename, and the optimizer-compatible `score` column.

Control the blend with the variables at the top of the scoring cell: `BLEND_MEAN_POINTS = True` enables it, `False` disables it, and `None` asks via Python `input()` each run (the default). Set `CALCULATED_EXPECTED_POINTS_WEIGHT` and `MEAN_POINTS_WEIGHT` to change the split; both must be between 0 and 1 and sum to 1. Defaults are 0.8 and 0.2. When enabled, `score = CALCULATED_EXPECTED_POINTS_WEIGHT * calculated expected points + MEAN_POINTS_WEIGHT * recency-weighted mean points`. At the prompt, answer `y` or `yes` to enable the configured blend; answer `n`, `no`, or press Enter to keep the original calculation. The mean uses per-match derived points from the same eligible appearances (at least 10 minutes) as the percentile ratings, with the same 28/24/20/16/12 team-match weights and missing-slot renormalization. It is not a points-per-90 average and is not rounded before blending. Goalkeepers use the calibrated points already stored in the percentile export. The mean contribution is added after odds and lineup scoring, so it still contributes when starting chance is zero; players without matched history retain zero. Enabled exports with a positive mean-points weight also include `calculated_expected_points` and `recency_weighted_mean_points`. Rerun the scoring cell to apply the variables and create a new CSV; saved outputs below reflect the previous run.


Goalkeeper expected match points are `1.1 + raw expected match points * (0.6 / 3)`: raw 0 maps to 1.1, 1.5 maps to 1.4, and 3 maps to 1.7. Outfield players retain raw 0–3 values. The exported `expected_match_points` is the value used in the score. Rerun this notebook to generate updated scores.

Goalkeeper alternatives use decay **0.60** (relative weights **1, 0.6, 0.36, …**), with existing provider normalization and injury handling. Outfield alternatives retain their configured decay.


In [1]:
import sys
from pathlib import Path
from IPython.display import display
import pandas as pd

def locate_project_root():
    for start in (Path.cwd().resolve(), Path(globals().get('__vsc_ipynb_file__', '.')).resolve()):
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Run from within the project directory.')
PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import sofascore_rating_odds_lineup_score as shared_scoring
import sofascore_kickbase_percentile_ratings as percentile_ratings
importlib.reload(shared_scoring)
importlib.reload(percentile_ratings)


<module 'sofascore_kickbase_percentile_ratings' from 'C:\\kickbase project\\sofascore_kickbase_percentile_ratings.py'>

In [2]:





































































































































































# Refresh both modules so an existing Jupyter kernel uses the latest scoring code.
import importlib
import sofascore_rating_odds_lineup_score as shared_scoring
import sofascore_kickbase_percentile_ratings as percentile_ratings
importlib.reload(shared_scoring)
importlib.reload(percentile_ratings)

QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY = 0.15
ALTERNATIVE_STARTING_CHANCE_DECAY = 0.45
GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY = 0.60
LINEUP_SOURCE_WEIGHTS = {'ligainsider': 4.0, 'kickbase': 3.0, 'kicker': 2.0, 'rotowire': 1.0}
SCORE_MULTIPLIER = 10.0062
BLEND_MEAN_POINTS = None  # True: enable; False: disable; None: ask using input()
CALCULATED_EXPECTED_POINTS_WEIGHT = 0.80
MEAN_POINTS_WEIGHT = 0.20  # The two weights must sum to 1

use_mean_points_blend = BLEND_MEAN_POINTS
if use_mean_points_blend is None:
    while True:
        blend_choice = input(f'Blend expected points with recency-weighted mean points ({CALCULATED_EXPECTED_POINTS_WEIGHT:.0%}/{MEAN_POINTS_WEIGHT:.0%})? [y/N]: ').strip().lower()
        if blend_choice in {'', 'n', 'no', 'y', 'yes'}:
            use_mean_points_blend = blend_choice in {'y', 'yes'}
            break
        print('Please enter y or n.')
result = percentile_ratings.run_score_creation(
    questionable_injury_starting_chance_penalty=QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY,
    alternative_starting_chance_decay=ALTERNATIVE_STARTING_CHANCE_DECAY,
    goalkeeper_alternative_starting_chance_decay=GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY,
    lineup_source_weights=LINEUP_SOURCE_WEIGHTS,
    score_multiplier=SCORE_MULTIPLIER,
    blend_mean_points=use_mean_points_blend,
    calculated_expected_points_weight=CALCULATED_EXPECTED_POINTS_WEIGHT,
    mean_points_weight=MEAN_POINTS_WEIGHT,
)


Blend expected points with recency-weighted mean points (80%/20%)? [y/N]:  y
Matchday for odds and lineups:  6
Is the Ligainsider predicted lineup snapshot (ligainsider_bundesliga_lineups_20261016_113334.json) accurate for matchday 6, or from the previous matchday? [a]ccurate/[p]revious:  a
Is the Kickbase predicted lineup snapshot (kickbase_bundesliga_lineups_20261016_120556.json) accurate for matchday 6, or from the previous matchday? [a]ccurate/[p]revious:  a
Is the Kicker predicted lineup snapshot (kicker_bundesliga_lineups_20261016_113641.json) accurate for matchday 6, or from the previous matchday? [a]ccurate/[p]revious:  a
Is the Rotowire predicted lineup snapshot (rotowire_bundesliga_lineups_20261016_113310.json) accurate for matchday 6, or from the previous matchday? [a]ccurate/[p]revious:  a



No exact match for ligainsider lineup for Lucas Höler (freiburg)
  1. Lucas Hoeler (ID 6593, similarity 95%)


Choose a candidate number, or press Enter to skip (score 0):  1



No exact match for kickbase lineup for Nicolas Kühn (gladbach)
  1. NICOLAS (displayed name, similarity 78%)


Choose a candidate number, or press Enter to skip (score 0):  


...

Choose a candidate number, or press Enter to skip (score 0):  


Input summary
  Matchday: 6; Kickbase percentile rating: sofascore_kickbase_percentile_ratings_2026-10-16_11-22-48_017318+0200.json
  Ligainsider (weight 4; current): ligainsider_bundesliga_lineups_20261016_113334.json
  Kickbase (weight 3; current): kickbase_bundesliga_lineups_20261016_120556.json
  Kicker (weight 2; current): kicker_bundesliga_lineups_20261016_113641.json
  Rotowire (weight 1; current): rotowire_bundesliga_lineups_20261016_113310.json
  Alternative starting-chance decay: 0.45
  Questionable-injury starting-chance penalty: 0.15
  Score multiplier: 10.0062
  Expected-points blend: 80% calculated + 20% recency-weighted mean.
  Output: C:\kickbase project\outputs\expected_points\expected_points_20261016_111610_+0200_sofascore_kickbase_percentile_rating_odds_lineup_20261016_122716_+0200.csv

Team odds and expected match points


,team,provider,win_probability,draw_probability,loss_probability,expected_match_points
0,augsburg,FotMob,0.357640,0.263262,0.379098,1.336181
1,bayern,FotMob,0.778612,0.115842,0.105545,2.451680
2,bremen,FotMob,0.542448,0.226020,0.231533,1.853363
3,dortmund,FotMob,0.585701,0.220659,0.193640,1.977762
4,elversberg,FotMob,0.379098,0.263262,0.357640,1.400556
5,frankfurt,FotMob,0.498437,0.230983,0.270580,1.726294
6,freiburg,FotMob,0.215710,0.225982,0.558308,0.873112
7,gladbach,FotMob,0.294118,0.235294,0.470588,1.117647
8,hamburg,FotMob,0.242843,0.230997,0.526160,0.959526
9,hoffenheim,FotMob,0.470588,0.235294,0.294118,1.647059



Player score review (non-positive or non-exact historical metric)


,name,team,metric_status,lineup_status,kickbase_percentile_rating,expected_match_points,ligainsider_starting_chance,kickbase_starting_chance,kicker_starting_chance,rotowire_starting_chance,questionable_injury_penalty,starting_chance,score_multiplier,score,calculated_expected_points,recency_weighted_mean_points
0,Jonathan Tah,bayern,missing_history,ligainsider:exact; kickbase:normalized; kicker...,8.2,2.451680,1.000000,0.689655,1.0,1.0,0.00,0.906897,10.0062,169.708673,182.433461,118.809524
1,Manuel Neuer,bayern,missing_history,ligainsider:exact; kickbase:normalized; kicker...,6.9,1.590336,0.264706,0.375000,0.0,0.0,0.15,0.218382,10.0062,39.925775,23.978647,103.714286
2,Sven Ulreich,bayern,missing_history,ligainsider:missing; kickbase:missing; kicker:...,0.0,1.590336,0.000000,0.000000,0.0,0.0,0.00,0.000000,10.0062,0.000000,0.000000,0.000000
3,Joshua Kimmich,bayern,missing_history,ligainsider:exact; kickbase:normalized; kicker...,9.3,2.451680,1.000000,1.000000,1.0,1.0,0.00,1.000000,10.0062,218.558054,228.147567,180.200000
4,Serge Gnabry,bayern,missing_history,ligainsider:exact; kickbase:normalized; kicker...,0.0,2.451680,0.689655,0.689655,1.0,0.0,0.00,0.682759,10.0062,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460,Nikola Katic,schalke,missing_history,ligainsider:exact; kickbase:normalized; kicker...,8.0,1.210526,1.000000,1.000000,1.0,1.0,0.00,1.000000,10.0062,104.021718,96.902147,132.500000
461,Luca Vozar,schalke,missing_history,ligainsider:missing; kickbase:normalized; kick...,0.0,1.210526,0.000000,0.272315,0.0,0.0,0.00,0.081694,10.0062,0.000000,0.000000,0.000000
462,Satoshi Tanaka,schalke,missing_history,ligainsider:exact; kickbase:normalized; kicker...,5.4,1.210526,0.689655,0.605144,1.0,1.0,0.00,0.757405,10.0062,48.490005,49.541078,44.285714
463,Johannes Simon Siebeking,schalke,missing_history,ligainsider:missing; kickbase:missing; kicker:...,0.0,1.342105,0.000000,0.000000,0.0,0.0,0.00,0.000000,10.0062,0.000000,0.000000,0.000000
